In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler, MinMaxScaler
from sklearn.metrics import accuracy_score 
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
from sklearn.metrics import make_scorer
from xgboost import XGBClassifier
    

In [2]:
train = pd.read_csv('/kaggle/input/child-mind-institute-problematic-internet-use/train.csv')
train

,id,Basic_Demos-Enroll_Season,Basic_Demos-Age,Basic_Demos-Sex,CGAS-Season,CGAS-CGAS_Score,Physical-Season,Physical-BMI,Physical-Height,Physical-Weight,...,PCIAT-PCIAT_18,PCIAT-PCIAT_19,PCIAT-PCIAT_20,PCIAT-PCIAT_Total,SDS-Season,SDS-SDS_Total_Raw,SDS-SDS_Total_T,PreInt_EduHx-Season,PreInt_EduHx-computerinternet_hoursday,sii
0,00008ff9,Fall,5,0,Winter,51.0,Fall,16.877316,46.0,50.8,...,4.0,2.0,4.0,55.0,NaN,NaN,NaN,Fall,3.0,2.0
1,000fd460,Summer,9,0,NaN,NaN,Fall,14.035590,48.0,46.0,...,0.0,0.0,0.0,0.0,Fall,46.0,64.0,Summer,0.0,0.0
2,00105258,Summer,10,1,Fall,71.0,Fall,16.648696,56.5,75.6,...,2.0,1.0,1.0,28.0,Fall,38.0,54.0,Summer,2.0,0.0
3,00115b9f,Winter,9,0,Fall,71.0,Summer,18.292347,56.0,81.6,...,3.0,4.0,1.0,44.0,Summer,31.0,45.0,Winter,0.0,1.0
4,0016bb22,Spring,18,1,Summer,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3955,ff8a2de4,Fall,13,0,Spring,60.0,Fall,16.362460,59.5,82.4,...,1.0,1.0,0.0,32.0,Winter,35.0,50.0,Fall,1.0,1.0
3956,ffa9794a,Winter,10,0,NaN,NaN,Spring,18.764678,53.5,76.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Winter,0.0,NaN
3957,ffcd4dbd,Fall,11,0,Spring,68.0,Winter,21.441500,60.0,109.8,...,1.0,0.0,1.0,31.0,Winter,56.0,77.0,Fall,0.0,1.0
3958,ffed1dd5,Spring,13,0,Spring,70.0,Winter,12.235895,70.7,87.0,...,1.0,1.0,1.0,19.0,Spring,33.0,47.0,Spring,1.0,0.0


In [3]:
test = pd.read_csv('/kaggle/input/child-mind-institute-problematic-internet-use/test.csv')
test 

,id,Basic_Demos-Enroll_Season,Basic_Demos-Age,Basic_Demos-Sex,CGAS-Season,CGAS-CGAS_Score,Physical-Season,Physical-BMI,Physical-Height,Physical-Weight,...,BIA-BIA_TBW,PAQ_A-Season,PAQ_A-PAQ_A_Total,PAQ_C-Season,PAQ_C-PAQ_C_Total,SDS-Season,SDS-SDS_Total_Raw,SDS-SDS_Total_T,PreInt_EduHx-Season,PreInt_EduHx-computerinternet_hoursday
0,00008ff9,Fall,5,0,Winter,51.0,Fall,16.877316,46.00,50.8,...,32.6909,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fall,3.0
1,000fd460,Summer,9,0,NaN,NaN,Fall,14.035590,48.00,46.0,...,27.0552,NaN,NaN,Fall,2.340,Fall,46.0,64.0,Summer,0.0
2,00105258,Summer,10,1,Fall,71.0,Fall,16.648696,56.50,75.6,...,NaN,NaN,NaN,Summer,2.170,Fall,38.0,54.0,Summer,2.0
3,00115b9f,Winter,9,0,Fall,71.0,Summer,18.292347,56.00,81.6,...,45.9966,NaN,NaN,Winter,2.451,Summer,31.0,45.0,Winter,0.0
4,0016bb22,Spring,18,1,Summer,NaN,NaN,NaN,NaN,NaN,...,NaN,Summer,1.04,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,001f3379,Spring,13,1,Winter,50.0,Summer,22.279952,59.50,112.2,...,63.1265,NaN,NaN,Spring,4.110,Summer,40.0,56.0,Spring,0.0
6,0038ba98,Fall,10,0,NaN,NaN,Fall,19.660760,55.00,84.6,...,47.2211,NaN,NaN,Winter,3.670,Winter,27.0,40.0,Fall,3.0
7,0068a485,Fall,10,1,NaN,NaN,Fall,16.861286,59.25,84.2,...,50.4767,NaN,NaN,Fall,1.270,NaN,NaN,NaN,Fall,2.0
8,0069fbed,Summer,15,0,NaN,NaN,Spring,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Summer,2.0
9,0083e397,Summer,19,1,Summer,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
threshold = 0.5 * len(train)
column_with_data = train.columns[train.isnull().sum() < threshold]
train = train[column_with_data]

In [5]:
train

,id,Basic_Demos-Enroll_Season,Basic_Demos-Age,Basic_Demos-Sex,CGAS-Season,CGAS-CGAS_Score,Physical-Season,Physical-BMI,Physical-Height,Physical-Weight,...,PCIAT-PCIAT_18,PCIAT-PCIAT_19,PCIAT-PCIAT_20,PCIAT-PCIAT_Total,SDS-Season,SDS-SDS_Total_Raw,SDS-SDS_Total_T,PreInt_EduHx-Season,PreInt_EduHx-computerinternet_hoursday,sii
0,00008ff9,Fall,5,0,Winter,51.0,Fall,16.877316,46.0,50.8,...,4.0,2.0,4.0,55.0,NaN,NaN,NaN,Fall,3.0,2.0
1,000fd460,Summer,9,0,NaN,NaN,Fall,14.035590,48.0,46.0,...,0.0,0.0,0.0,0.0,Fall,46.0,64.0,Summer,0.0,0.0
2,00105258,Summer,10,1,Fall,71.0,Fall,16.648696,56.5,75.6,...,2.0,1.0,1.0,28.0,Fall,38.0,54.0,Summer,2.0,0.0
3,00115b9f,Winter,9,0,Fall,71.0,Summer,18.292347,56.0,81.6,...,3.0,4.0,1.0,44.0,Summer,31.0,45.0,Winter,0.0,1.0
4,0016bb22,Spring,18,1,Summer,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3955,ff8a2de4,Fall,13,0,Spring,60.0,Fall,16.362460,59.5,82.4,...,1.0,1.0,0.0,32.0,Winter,35.0,50.0,Fall,1.0,1.0
3956,ffa9794a,Winter,10,0,NaN,NaN,Spring,18.764678,53.5,76.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Winter,0.0,NaN
3957,ffcd4dbd,Fall,11,0,Spring,68.0,Winter,21.441500,60.0,109.8,...,1.0,0.0,1.0,31.0,Winter,56.0,77.0,Fall,0.0,1.0
3958,ffed1dd5,Spring,13,0,Spring,70.0,Winter,12.235895,70.7,87.0,...,1.0,1.0,1.0,19.0,Spring,33.0,47.0,Spring,1.0,0.0


In [6]:
train = train.dropna(subset = 'sii')

train

,id,Basic_Demos-Enroll_Season,Basic_Demos-Age,Basic_Demos-Sex,CGAS-Season,CGAS-CGAS_Score,Physical-Season,Physical-BMI,Physical-Height,Physical-Weight,...,PCIAT-PCIAT_18,PCIAT-PCIAT_19,PCIAT-PCIAT_20,PCIAT-PCIAT_Total,SDS-Season,SDS-SDS_Total_Raw,SDS-SDS_Total_T,PreInt_EduHx-Season,PreInt_EduHx-computerinternet_hoursday,sii
0,00008ff9,Fall,5,0,Winter,51.0,Fall,16.877316,46.0,50.8,...,4.0,2.0,4.0,55.0,NaN,NaN,NaN,Fall,3.0,2.0
1,000fd460,Summer,9,0,NaN,NaN,Fall,14.035590,48.0,46.0,...,0.0,0.0,0.0,0.0,Fall,46.0,64.0,Summer,0.0,0.0
2,00105258,Summer,10,1,Fall,71.0,Fall,16.648696,56.5,75.6,...,2.0,1.0,1.0,28.0,Fall,38.0,54.0,Summer,2.0,0.0
3,00115b9f,Winter,9,0,Fall,71.0,Summer,18.292347,56.0,81.6,...,3.0,4.0,1.0,44.0,Summer,31.0,45.0,Winter,0.0,1.0
5,001f3379,Spring,13,1,Winter,50.0,Summer,22.279952,59.5,112.2,...,1.0,2.0,1.0,34.0,Summer,40.0,56.0,Spring,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3953,ff6c2bb8,Fall,8,0,NaN,NaN,Fall,17.139810,52.5,67.2,...,2.0,2.0,1.0,22.0,Fall,41.0,58.0,Fall,2.0,0.0
3954,ff759544,Summer,7,1,NaN,NaN,Summer,13.927006,48.5,46.6,...,3.0,3.0,0.0,33.0,Summer,48.0,67.0,Summer,0.0,1.0
3955,ff8a2de4,Fall,13,0,Spring,60.0,Fall,16.362460,59.5,82.4,...,1.0,1.0,0.0,32.0,Winter,35.0,50.0,Fall,1.0,1.0
3957,ffcd4dbd,Fall,11,0,Spring,68.0,Winter,21.441500,60.0,109.8,...,1.0,0.0,1.0,31.0,Winter,56.0,77.0,Fall,0.0,1.0


In [7]:
columns_to_fill = [col for col in train.columns if "-Season" not in col]

train[columns_to_fill] = train[columns_to_fill].fillna(-0.1)

train

/tmp/ipykernel_17/2997305297.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train[columns_to_fill] = train[columns_to_fill].fillna(-0.1)


,id,Basic_Demos-Enroll_Season,Basic_Demos-Age,Basic_Demos-Sex,CGAS-Season,CGAS-CGAS_Score,Physical-Season,Physical-BMI,Physical-Height,Physical-Weight,...,PCIAT-PCIAT_18,PCIAT-PCIAT_19,PCIAT-PCIAT_20,PCIAT-PCIAT_Total,SDS-Season,SDS-SDS_Total_Raw,SDS-SDS_Total_T,PreInt_EduHx-Season,PreInt_EduHx-computerinternet_hoursday,sii
0,00008ff9,Fall,5,0,Winter,51.0,Fall,16.877316,46.0,50.8,...,4.0,2.0,4.0,55.0,NaN,-0.1,-0.1,Fall,3.0,2.0
1,000fd460,Summer,9,0,NaN,-0.1,Fall,14.035590,48.0,46.0,...,0.0,0.0,0.0,0.0,Fall,46.0,64.0,Summer,0.0,0.0
2,00105258,Summer,10,1,Fall,71.0,Fall,16.648696,56.5,75.6,...,2.0,1.0,1.0,28.0,Fall,38.0,54.0,Summer,2.0,0.0
3,00115b9f,Winter,9,0,Fall,71.0,Summer,18.292347,56.0,81.6,...,3.0,4.0,1.0,44.0,Summer,31.0,45.0,Winter,0.0,1.0
5,001f3379,Spring,13,1,Winter,50.0,Summer,22.279952,59.5,112.2,...,1.0,2.0,1.0,34.0,Summer,40.0,56.0,Spring,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3953,ff6c2bb8,Fall,8,0,NaN,-0.1,Fall,17.139810,52.5,67.2,...,2.0,2.0,1.0,22.0,Fall,41.0,58.0,Fall,2.0,0.0
3954,ff759544,Summer,7,1,NaN,-0.1,Summer,13.927006,48.5,46.6,...,3.0,3.0,0.0,33.0,Summer,48.0,67.0,Summer,0.0,1.0
3955,ff8a2de4,Fall,13,0,Spring,60.0,Fall,16.362460,59.5,82.4,...,1.0,1.0,0.0,32.0,Winter,35.0,50.0,Fall,1.0,1.0
3957,ffcd4dbd,Fall,11,0,Spring,68.0,Winter,21.441500,60.0,109.8,...,1.0,0.0,1.0,31.0,Winter,56.0,77.0,Fall,0.0,1.0


In [8]:
numeric_data = train.select_dtypes(include=["number"])
numeric_data = numeric_data.drop(columns = ['sii'])
correlation_matrix = numeric_data.corr()
high_corr_pairs = correlation_matrix[(correlation_matrix > 0.75) & (correlation_matrix < 1)]
columns_to_drop = set()
selected_columns = set()
for column in high_corr_pairs.columns:
    if column not in columns_to_drop and column not in selected_columns:
        # Lấy các cột tương quan cao với cột hiện tại
        correlated_columns = high_corr_pairs.index[high_corr_pairs[column].notnull()].tolist()
        
        # Thêm các cột tương quan vào danh sách loại bỏ
        columns_to_drop.update(correlated_columns)
        
        # Giữ lại cột hiện tại
        selected_columns.add(column)

# Loại bỏ các cột dư thừa trong dữ liệu gốc
columns_to_keep = [col for col in train.columns if col not in columns_to_drop]
train = train[columns_to_keep]

In [9]:
train

,id,Basic_Demos-Enroll_Season,Basic_Demos-Age,Basic_Demos-Sex,CGAS-Season,CGAS-CGAS_Score,Physical-Season,Physical-BMI,Physical-Diastolic_BP,Physical-HeartRate,...,PCIAT-PCIAT_15,PCIAT-PCIAT_16,PCIAT-PCIAT_17,PCIAT-PCIAT_19,PCIAT-PCIAT_20,SDS-Season,SDS-SDS_Total_Raw,PreInt_EduHx-Season,PreInt_EduHx-computerinternet_hoursday,sii
0,00008ff9,Fall,5,0,Winter,51.0,Fall,16.877316,-0.1,-0.1,...,4.0,4.0,4.0,2.0,4.0,NaN,-0.1,Fall,3.0,2.0
1,000fd460,Summer,9,0,NaN,-0.1,Fall,14.035590,75.0,70.0,...,0.0,0.0,0.0,0.0,0.0,Fall,46.0,Summer,0.0,0.0
2,00105258,Summer,10,1,Fall,71.0,Fall,16.648696,65.0,94.0,...,1.0,0.0,2.0,1.0,1.0,Fall,38.0,Summer,2.0,0.0
3,00115b9f,Winter,9,0,Fall,71.0,Summer,18.292347,60.0,97.0,...,0.0,3.0,4.0,4.0,1.0,Summer,31.0,Winter,0.0,1.0
5,001f3379,Spring,13,1,Winter,50.0,Summer,22.279952,60.0,73.0,...,2.0,1.0,3.0,2.0,1.0,Summer,40.0,Spring,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3953,ff6c2bb8,Fall,8,0,NaN,-0.1,Fall,17.139810,60.0,65.0,...,0.0,3.0,0.0,2.0,1.0,Fall,41.0,Fall,2.0,0.0
3954,ff759544,Summer,7,1,NaN,-0.1,Summer,13.927006,65.0,75.0,...,0.0,5.0,3.0,3.0,0.0,Summer,48.0,Summer,0.0,1.0
3955,ff8a2de4,Fall,13,0,Spring,60.0,Fall,16.362460,71.0,70.0,...,1.0,0.0,2.0,1.0,0.0,Winter,35.0,Fall,1.0,1.0
3957,ffcd4dbd,Fall,11,0,Spring,68.0,Winter,21.441500,79.0,99.0,...,0.0,0.0,1.0,0.0,1.0,Winter,56.0,Fall,0.0,1.0


In [10]:
columns_to_fill = [col for col in test.columns if "-Season" not in col]

test[columns_to_fill] = test[columns_to_fill].fillna(-0.1)

test

,id,Basic_Demos-Enroll_Season,Basic_Demos-Age,Basic_Demos-Sex,CGAS-Season,CGAS-CGAS_Score,Physical-Season,Physical-BMI,Physical-Height,Physical-Weight,...,BIA-BIA_TBW,PAQ_A-Season,PAQ_A-PAQ_A_Total,PAQ_C-Season,PAQ_C-PAQ_C_Total,SDS-Season,SDS-SDS_Total_Raw,SDS-SDS_Total_T,PreInt_EduHx-Season,PreInt_EduHx-computerinternet_hoursday
0,00008ff9,Fall,5,0,Winter,51.0,Fall,16.877316,46.00,50.8,...,32.6909,NaN,-0.10,NaN,-0.100,NaN,-0.1,-0.1,Fall,3.0
1,000fd460,Summer,9,0,NaN,-0.1,Fall,14.035590,48.00,46.0,...,27.0552,NaN,-0.10,Fall,2.340,Fall,46.0,64.0,Summer,0.0
2,00105258,Summer,10,1,Fall,71.0,Fall,16.648696,56.50,75.6,...,-0.1000,NaN,-0.10,Summer,2.170,Fall,38.0,54.0,Summer,2.0
3,00115b9f,Winter,9,0,Fall,71.0,Summer,18.292347,56.00,81.6,...,45.9966,NaN,-0.10,Winter,2.451,Summer,31.0,45.0,Winter,0.0
4,0016bb22,Spring,18,1,Summer,-0.1,NaN,-0.100000,-0.10,-0.1,...,-0.1000,Summer,1.04,NaN,-0.100,NaN,-0.1,-0.1,NaN,-0.1
5,001f3379,Spring,13,1,Winter,50.0,Summer,22.279952,59.50,112.2,...,63.1265,NaN,-0.10,Spring,4.110,Summer,40.0,56.0,Spring,0.0
6,0038ba98,Fall,10,0,NaN,-0.1,Fall,19.660760,55.00,84.6,...,47.2211,NaN,-0.10,Winter,3.670,Winter,27.0,40.0,Fall,3.0
7,0068a485,Fall,10,1,NaN,-0.1,Fall,16.861286,59.25,84.2,...,50.4767,NaN,-0.10,Fall,1.270,NaN,-0.1,-0.1,Fall,2.0
8,0069fbed,Summer,15,0,NaN,-0.1,Spring,-0.100000,-0.10,-0.1,...,-0.1000,NaN,-0.10,NaN,-0.100,NaN,-0.1,-0.1,Summer,2.0
9,0083e397,Summer,19,1,Summer,-0.1,NaN,-0.100000,-0.10,-0.1,...,-0.1000,NaN,-0.10,NaN,-0.100,NaN,-0.1,-0.1,NaN,-0.1


In [11]:
train = train.drop(columns = ["id"])


In [12]:
train

,Basic_Demos-Enroll_Season,Basic_Demos-Age,Basic_Demos-Sex,CGAS-Season,CGAS-CGAS_Score,Physical-Season,Physical-BMI,Physical-Diastolic_BP,Physical-HeartRate,FGC-Season,...,PCIAT-PCIAT_15,PCIAT-PCIAT_16,PCIAT-PCIAT_17,PCIAT-PCIAT_19,PCIAT-PCIAT_20,SDS-Season,SDS-SDS_Total_Raw,PreInt_EduHx-Season,PreInt_EduHx-computerinternet_hoursday,sii
0,Fall,5,0,Winter,51.0,Fall,16.877316,-0.1,-0.1,Fall,...,4.0,4.0,4.0,2.0,4.0,NaN,-0.1,Fall,3.0,2.0
1,Summer,9,0,NaN,-0.1,Fall,14.035590,75.0,70.0,Fall,...,0.0,0.0,0.0,0.0,0.0,Fall,46.0,Summer,0.0,0.0
2,Summer,10,1,Fall,71.0,Fall,16.648696,65.0,94.0,Fall,...,1.0,0.0,2.0,1.0,1.0,Fall,38.0,Summer,2.0,0.0
3,Winter,9,0,Fall,71.0,Summer,18.292347,60.0,97.0,Summer,...,0.0,3.0,4.0,4.0,1.0,Summer,31.0,Winter,0.0,1.0
5,Spring,13,1,Winter,50.0,Summer,22.279952,60.0,73.0,Summer,...,2.0,1.0,3.0,2.0,1.0,Summer,40.0,Spring,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3953,Fall,8,0,NaN,-0.1,Fall,17.139810,60.0,65.0,Fall,...,0.0,3.0,0.0,2.0,1.0,Fall,41.0,Fall,2.0,0.0
3954,Summer,7,1,NaN,-0.1,Summer,13.927006,65.0,75.0,Summer,...,0.0,5.0,3.0,3.0,0.0,Summer,48.0,Summer,0.0,1.0
3955,Fall,13,0,Spring,60.0,Fall,16.362460,71.0,70.0,Fall,...,1.0,0.0,2.0,1.0,0.0,Winter,35.0,Fall,1.0,1.0
3957,Fall,11,0,Spring,68.0,Winter,21.441500,79.0,99.0,Winter,...,0.0,0.0,1.0,0.0,1.0,Winter,56.0,Fall,0.0,1.0


In [13]:
season_columns = ['Basic_Demos-Enroll_Season', 'CGAS-Season', 'Physical-Season',
                  'FGC-Season', 'BIA-Season', 'SDS-Season', 'PreInt_EduHx-Season'
                 ]

In [14]:
test

,id,Basic_Demos-Enroll_Season,Basic_Demos-Age,Basic_Demos-Sex,CGAS-Season,CGAS-CGAS_Score,Physical-Season,Physical-BMI,Physical-Height,Physical-Weight,...,BIA-BIA_TBW,PAQ_A-Season,PAQ_A-PAQ_A_Total,PAQ_C-Season,PAQ_C-PAQ_C_Total,SDS-Season,SDS-SDS_Total_Raw,SDS-SDS_Total_T,PreInt_EduHx-Season,PreInt_EduHx-computerinternet_hoursday
0,00008ff9,Fall,5,0,Winter,51.0,Fall,16.877316,46.00,50.8,...,32.6909,NaN,-0.10,NaN,-0.100,NaN,-0.1,-0.1,Fall,3.0
1,000fd460,Summer,9,0,NaN,-0.1,Fall,14.035590,48.00,46.0,...,27.0552,NaN,-0.10,Fall,2.340,Fall,46.0,64.0,Summer,0.0
2,00105258,Summer,10,1,Fall,71.0,Fall,16.648696,56.50,75.6,...,-0.1000,NaN,-0.10,Summer,2.170,Fall,38.0,54.0,Summer,2.0
3,00115b9f,Winter,9,0,Fall,71.0,Summer,18.292347,56.00,81.6,...,45.9966,NaN,-0.10,Winter,2.451,Summer,31.0,45.0,Winter,0.0
4,0016bb22,Spring,18,1,Summer,-0.1,NaN,-0.100000,-0.10,-0.1,...,-0.1000,Summer,1.04,NaN,-0.100,NaN,-0.1,-0.1,NaN,-0.1
5,001f3379,Spring,13,1,Winter,50.0,Summer,22.279952,59.50,112.2,...,63.1265,NaN,-0.10,Spring,4.110,Summer,40.0,56.0,Spring,0.0
6,0038ba98,Fall,10,0,NaN,-0.1,Fall,19.660760,55.00,84.6,...,47.2211,NaN,-0.10,Winter,3.670,Winter,27.0,40.0,Fall,3.0
7,0068a485,Fall,10,1,NaN,-0.1,Fall,16.861286,59.25,84.2,...,50.4767,NaN,-0.10,Fall,1.270,NaN,-0.1,-0.1,Fall,2.0
8,0069fbed,Summer,15,0,NaN,-0.1,Spring,-0.100000,-0.10,-0.1,...,-0.1000,NaN,-0.10,NaN,-0.100,NaN,-0.1,-0.1,Summer,2.0
9,0083e397,Summer,19,1,Summer,-0.1,NaN,-0.100000,-0.10,-0.1,...,-0.1000,NaN,-0.10,NaN,-0.100,NaN,-0.1,-0.1,NaN,-0.1


In [15]:
OHE = OneHotEncoder(handle_unknown = 'ignore', sparse_output= False).set_output(transform = 'pandas')

In [16]:
Season_OHE = OHE.fit_transform(train[season_columns])

In [17]:
x = train.drop(columns='sii')
y = train['sii']

In [18]:
common_columns = x.columns.intersection(test.columns)
x_test = x[common_columns]

In [19]:
common_columns = x.columns.intersection(x_test.columns)
x_test = x[common_columns]

In [20]:
x_test

,Basic_Demos-Enroll_Season,Basic_Demos-Age,Basic_Demos-Sex,CGAS-Season,CGAS-CGAS_Score,Physical-Season,Physical-BMI,Physical-Diastolic_BP,Physical-HeartRate,FGC-Season,...,BIA-Season,BIA-BIA_Activity_Level_num,BIA-BIA_BMC,BIA-BIA_FFMI,BIA-BIA_FMI,BIA-BIA_Fat,SDS-Season,SDS-SDS_Total_Raw,PreInt_EduHx-Season,PreInt_EduHx-computerinternet_hoursday
0,Fall,5,0,Winter,51.0,Fall,16.877316,-0.1,-0.1,Fall,...,Fall,2.0,2.66855,13.8177,3.061430,9.21377,NaN,-0.1,Fall,3.0
1,Summer,9,0,NaN,-0.1,Fall,14.035590,75.0,70.0,Fall,...,Winter,2.0,2.57949,12.8254,1.211720,3.97085,Fall,46.0,Summer,0.0
2,Summer,10,1,Fall,71.0,Fall,16.648696,65.0,94.0,Fall,...,NaN,-0.1,-0.10000,-0.1000,-0.100000,-0.10000,Fall,38.0,Summer,2.0
3,Winter,9,0,Fall,71.0,Summer,18.292347,60.0,97.0,Summer,...,Summer,3.0,3.84191,14.0740,4.220330,18.82430,Summer,31.0,Winter,0.0
5,Spring,13,1,Winter,50.0,Summer,22.279952,60.0,73.0,Summer,...,Summer,2.0,4.33036,16.6877,13.498800,67.97150,Summer,40.0,Spring,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3953,Fall,8,0,NaN,-0.1,Fall,17.139810,60.0,65.0,Fall,...,Fall,3.0,3.20303,13.4004,3.741300,14.66690,Fall,41.0,Fall,2.0
3954,Summer,7,1,NaN,-0.1,Summer,13.927006,65.0,75.0,Summer,...,Fall,1.0,2.36680,13.2315,0.414263,1.41470,Summer,48.0,Summer,0.0
3955,Fall,13,0,Spring,60.0,Fall,16.362460,71.0,70.0,Fall,...,Fall,3.0,4.52277,14.0629,2.301380,11.58830,Winter,35.0,Fall,1.0
3957,Fall,11,0,Spring,68.0,Winter,21.441500,79.0,99.0,Winter,...,Winter,2.0,4.41305,14.8043,6.639520,33.99670,Winter,56.0,Fall,0.0


In [21]:
x_train = x
y_train = y

In [22]:
categorical_cols = [season_columns]
numerical_cols = x_test.drop(columns = season_columns, axis=1, errors = 'ignore').columns.tolist()

In [23]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), season_columns),
        ('num', RobustScaler(), numerical_cols)                        
    ]
)

In [24]:
pipeline = Pipeline(steps=[
    ('preprcessor', preprocessor ),
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42))
])


In [25]:
param_grid = {
    'classifier__n_estimators': [100],       
    'classifier__max_depth': [10],         
    'classifier__min_samples_split': [10],      
    'classifier__min_samples_leaf': [2],        
    'classifier__max_features': ['sqrt']
}


In [26]:
def quadratic_weighted_kappa(actual, predicted, n_labels):
    """
    Calculate the quadratic weighted kappa (QWK).

    Parameters:
    actual (list or numpy array): Array of actual labels.
    predicted (list or numpy array): Array of predicted labels.
    n_labels (int): Number of distinct labels.

    Returns:
    float: Quadratic weighted kappa.
    """
    # Ensure inputs are numpy arrays
    actual = np.asarray(actual, dtype=int)
    predicted = np.asarray(predicted, dtype=int)

    # Initialize the O matrix (observed histogram)
    O = np.zeros((n_labels, n_labels), dtype=np.float64)
    for a, p in zip(actual, predicted):
        O[a, p] += 1

    # Compute the histogram of actual and predicted labels
    actual_hist = np.sum(O, axis=1)
    predicted_hist = np.sum(O, axis=0)

    # Calculate the E matrix (expected outcomes)
    E = np.outer(actual_hist, predicted_hist) / np.sum(O)

    # Create the W matrix (weights)
    W = np.zeros((n_labels, n_labels), dtype=np.float64)
    for i in range(n_labels):
        for j in range(n_labels):
            W[i, j] = ((i - j) ** 2) / ((n_labels - 1) ** 2)

    # Calculate the quadratic weighted kappa
    numerator = np.sum(W * O)
    denominator = np.sum(W * E)

    kappa = 1 - numerator / denominator
    return kappa

def qwk_scorer(n_labels):
    """
    Create a scorer function for GridSearchCV that calculates QWK.

    Parameters:
    n_labels (int): Number of distinct labels.

    Returns:
    function: Scorer function for GridSearchCV.
    """
    def scorer(y_true, y_pred):
        return quadratic_weighted_kappa(y_true, y_pred, n_labels)

    return make_scorer(scorer, greater_is_better=True)

In [27]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring=qwk_scorer(4),
    verbose=2,
    n_jobs=-1
)


In [28]:
grid_search.fit(x_train, y_train)

Fitting 5 folds for each of 1 candidates, totalling 5 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprcessor',
                                        ColumnTransformer(transformers=[('cat',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['Basic_Demos-Enroll_Season',
                                                                          'CGAS-Season',
                                                                          'Physical-Season',
                                                                          'FGC-Season',
                                                                          'BIA-Season',
                                                                          'SDS-Season',
                                                                          'PreInt_EduHx-Season']),
                                                                        ('num',
                                                                         RobustScaler(),
                                                                         ['Basic_Demos-Age',
                                                                          'Basic_Demos-Sex',
                                                                          'CGAS-CGAS_Score',
                                                                          'Physical-BM...
                                                                          'PreInt_EduHx-computerinternet_hoursday'])])),
                                       ('classifier',
                                        RandomForestClassifier(class_weight='balanced',
                                                               random_state=42))]),
             n_jobs=-1,
             param_grid={'classifier__max_depth': [10],
                         'classifier__max_features': ['sqrt'],
                         'classifier__min_samples_leaf': [2],
                         'classifier__min_samples_split': [10],
                         'classifier__n_estimators': [100]},
             scoring=make_scorer(scorer), verbose=2)

In [29]:
grid_search.best_params_

{'classifier__max_depth': 10,
 'classifier__max_features': 'sqrt',
 'classifier__min_samples_leaf': 2,
 'classifier__min_samples_split': 10,
 'classifier__n_estimators': 100}

In [30]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(test.drop(columns = ['id']))

In [31]:
submission = pd.DataFrame({
    'id': test['id'],
    'sii': y_pred
}).set_index('id')
submission.to_csv('./submission.csv')

submission

,sii
id,
00008ff9,0.0
000fd460,0.0
00105258,0.0
00115b9f,0.0
0016bb22,2.0
001f3379,1.0
0038ba98,0.0
0068a485,0.0
0069fbed,2.0
